In [5]:
import sys
!{sys.executable} -m pip install statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 964.0 kB/s  0:00:09 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [statsmodels] [statsmodels]


In [37]:
import pandas as pd
import itertools
import statsmodels.api as sm
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [35]:
def exhaustive_regression(X, y, categorical_target=False):
    """
    Uses:
      - OLS for continuous targets
      - MNLogit for multiclass categorical targets

    Parameters
    ----------
    X : pd.DataFrame
        Feature matrix
    y : pd.Series
        Target vector

    Returns
    -------
    results : pd.DataFrame
        DataFrame with feature1, feature2, model type,
        interaction coefficients, and p-values.
    """
    feature_names = X.columns.tolist()

    # Detect target type
    if categorical_target:
        model_type = "multiclass"
        le = LabelEncoder()
        y = le.fit_transform(y)
        classes = le.classes_
    else:
        if pd.api.types.is_numeric_dtype(y):
            model_type = "continuous"
        else:
            model_type = "multiclass"
            le = LabelEncoder()
            y = le.fit_transform(y)
            classes = le.classes_

    results = []

    for f1, f2 in list(itertools.combinations(range(X.shape[1]), 2)):
        # Build design matrix
        feature_pair = X[[feature_names[f1], feature_names[f2]]].copy()
        feature_pair["interaction"] = X[feature_names[f1]] * X[feature_names[f2]]
        feature_pair = sm.add_constant(feature_pair)

        try: 

            if model_type == "continuous":
                # Ordinary Least Squares
                model = sm.OLS(y, feature_pair).fit()
                results.append({
                    "feature1": feature_names[f1],
                    "feature2": feature_names[f2],
                    "model": "OLS",
                    "interaction_coef": model.params.get("interaction", np.nan),
                    "interaction_pval": model.pvalues.get("interaction", np.nan)
                })
            else:
                # Multinomial Logistic Regression
                model = sm.MNLogit(y, feature_pair).fit(disp=False)
                if "interaction" in model.pvalues.index:
                    for class_idx, p in model.pvalues.loc["interaction"].items():
                        results.append({
                            "feature1": feature_names[f1],
                            "feature2": feature_names[f2],
                            "model": "MNLogit",
                            "class": classes[class_idx],
                            "interaction_coef": model.params.loc["interaction", class_idx],
                            "interaction_pval": p
                        })

        except Exception as e:
            results.append({
                "feature1": feature_names[f1],
                "feature2": feature_names[f2],
                "model": model_type,
                "error": str(e)
            })

    return pd.DataFrame(results)

TEST (no perturbation)

In [ ]:
# Load data
df = pd.read_csv("/Users/mariahloehr/IICD/IICD/Data/cell_cycle_tidied.csv")

# Define features and target
X = df.drop(columns=['phase', 'age', 'PHATE_1', 'PHATE_2'])  # Features
y = df['age']  # Target: age

In [20]:
# first 50 rows
X_small = X.iloc[:50, :]
y_small = y.iloc[:50]

# first 10 features
feature_subset = X.columns[:10] 
X_small = X[feature_subset].iloc[:50, :]
y_small = y.iloc[:50]

In [27]:
df = exhaustive_regression(X, y)

In [28]:
top10 = df.sort_values('interaction_pval').head(10)

In [29]:
top10

,feature1,feature2,model,interaction_coef,interaction_pval
4302,pRB..nuc.median.,p27..nuc.median.,OLS,-1.765153,0.000000e+00
6204,pp21..nuc.median.,p21..phospho.total.nuc.,OLS,-0.306167,8.255576e-292
5007,p27..nuc.median.,RB..phospho.total.nuc.,OLS,-1.552622,5.699296e-290
5029,p27..nuc.median.,ratio,OLS,-1.552622,5.699296e-290
6005,pp21..nuc.median.,pp65..nuc.median.,OLS,-1.068402,3.311855e-277
3320,RB..nuc.median.,p27..nuc.median.,OLS,-1.494585,7.761544e-263
6180,pp21..nuc.median.,ERK..phospho.total.cell.,OLS,1.852032,4.006552e-250
4795,p27..nuc.median.,DNA..nuc.median.,OLS,-1.535953,2.528862e-240
6224,pp21..nuc.median.,ratio,OLS,-2.926138,1.254877e-222
6202,pp21..nuc.median.,RB..phospho.total.nuc.,OLS,-2.926138,1.254877e-222


TEST (cancer)

In [39]:
# Load data
df = pd.read_csv("/Users/mariahloehr/IICD/IICD/Cancer treatment/T47D.csv")

# Separate features and target
X = df.drop(columns=['Metadata_well'])
X = X.select_dtypes(include=['number'])
y = df['Metadata_well']

In [38]:
df = exhaustive_regression(X, y, categorical_target=True)

In [ ]:
top10 = df.sort_values('interaction_pval').head(10)

In [ ]:
top10

,feature1,feature2,model,class,interaction_coef,interaction_pval
759,Intensity_MedianIntensity_pRB,pRB_over_RB,MNLogit,100,0.921525,0.0
671,Intensity_MedianIntensity_Skp2,Intensity_MedianIntensity_pRB,MNLogit,100,0.912245,0.0
670,Intensity_MedianIntensity_Skp2,Intensity_MedianIntensity_pRB,MNLogit,10,0.516427,0.0
695,Intensity_MedianIntensity_cycA,Intensity_MedianIntensity_pRB,MNLogit,100,0.785408,0.0
643,Intensity_MedianIntensity_RB,Intensity_MedianIntensity_pRB,MNLogit,100,1.124490,0.0
699,Intensity_MedianIntensity_cycA,pRB_over_RB,MNLogit,100,0.664718,0.0
395,Intensity_MedianIntensity_Cdh1,pRB_over_RB,MNLogit,100,0.853645,0.0
715,Intensity_MedianIntensity_cycB1,Intensity_MedianIntensity_pRB,MNLogit,100,0.564088,0.0
398,Intensity_MedianIntensity_Cdt1,Intensity_MedianIntensity_E2F1,MNLogit,10,0.433850,0.0
579,Intensity_MedianIntensity_Ki67,pRB_over_RB,MNLogit,100,1.191305,0.0
